# Creates cell masks by taking the CAAX channel, filling in holes, and removing unconnected objects

### Most useful for cells that are clumped together (hard to separate using the cell channel) with a strong CAAX+ cell border

In [ ]:
from pathlib import Path

from bioio import BioImage
import bioio_ome_tiff
from bioio.writers import OmeTiffWriter
import bioio_imageio

import numpy as np
import pandas as pd

%matplotlib notebook
%matplotlib inline
import matplotlib.pyplot as plt

import sys
src_path = str(Path.cwd().parent.parent)
if src_path not in sys.path:
    sys.path.append(src_path)
import src.d00_utils.utilities as utils
import src.d00_utils.dirnames as dn
from src.d01_init_proc import vis_and_rescale

from scipy import ndimage as ndi
from skimage.measure import label, regionprops

In [ ]:
df_path = Path(input())

In [ ]:
df = pd.read_csv(df_path)
df.head()

#### Input dirpath for images to stack with mask (to help with checking/manually editing mask files)

In [ ]:
ch = 1
ch_content = 'bgsub caax'

proc_dirpath = utils.get_proc_dirpath(df_path)

# will stack new masks with the 'caax cell stack' images to help with checking/manually editing mask files
caax_cell_dirpath = proc_dirpath / dn.stack_dirname
refinedmask_dirpath = proc_dirpath / dn.masks_dirname / dn.refinedmasks_dirname
fig_dirpath = refinedmask_dirpath / dn.figs_dirname
fig_dirpath.mkdir(parents=True, exist_ok=True)

df = df.rename(columns={'mask dirpath': 'polygonmask dirpath'})
df.insert(2, 'refinedmask dirpath', refinedmask_dirpath)
df.insert(3, 'caax cell dirpath', caax_cell_dirpath)

df.head()

In [ ]:
#TODO: move to applymask.py

def refine_binary_mask(bin_img, dtype='uint16'):
    
    # fill in holes
    filled_bin_img = ndi.binary_fill_holes(bin_img, axes=[3,4])
    
    # remove unconnected areas
    size_t = filled_bin_img.shape[0]
    filled_bin_filt = np.zeros_like(filled_bin_img)
    for t in range(size_t):
        labels_slice = label(filled_bin_img[t, 0, 0, :, :])
        regions = regionprops(labels_slice)
        largest_label = max(regions, key=lambda x: x.area).label
        filled_bin_filt[t, 0, 0, :, :] = labels_slice == largest_label

    # convert mask to 'uint8' format
    mask_refined = filled_bin_filt.astype(dtype)
    mask_refined = mask_refined * np.iinfo(dtype).max

    return mask_refined

In [ ]:
#TODO: delete
# prevdirpath = Path('/Users/kwu2/Library/CloudStorage/Box-Box/Z-lab Box/Z-lab shared folders/Kathryn + Eduardo/CE031/experiment/img_processing/masks/refinedmasks_prev')


for i, row in df.iterrows():

    if row[f'{ch_content} img'] is not np.nan:
        bgsub_path = Path(row[f'{ch_content} dirpath']) / row[f'{ch_content} img']
        maskpath = Path(row['polygonmask dirpath']) / row['mask name']
        caax_cell_path = Path(row['caax cell dirpath']) / (row['basename'] + '.ome.tif')

        if bgsub_path.is_file() & maskpath.is_file() & caax_cell_path.is_file():

            maskname_nosfx = maskpath.name.split('.png')[0]
            refinedmaskpath = refinedmask_dirpath / (maskname_nosfx + '.ome.tif')
            
            if not refinedmaskpath.is_file():

                print(f'Processing {maskname_nosfx}: {i}/{len(df)}')
                
                # get bgsub image
                bgsub_img_file = BioImage(bgsub_path, reader=bioio_ome_tiff.Reader)
                bgsub_img = bgsub_img_file.data
                
                # get mask
                polygonmask = np.array(BioImage(maskpath, reader=bioio_imageio.Reader).data)
        
                # refine mask
                bgsub_img_ch = bgsub_img[:, ch, np.newaxis, :, :, :]
                bin_img = np.where((bgsub_img_ch > 0) & (polygonmask > 0), 1, 0)
                mask_refined = refine_binary_mask(bin_img)

                # # TODO: delete
                # mask_refined = (((mask_refined + prevmask) > 0).astype('int') * np.iinfo('uint16').max).astype('uint16')
                # print(mask_refined.shape)
                
                # create and save figure for easy visualization
                imgs_to_vis = [bin_img, mask_refined]
                imglabels = [f'{ch_content}', 'refined mask']
                fig = vis_and_rescale.create_fig(imgs_to_vis, imglabels)
                fig.suptitle(maskname_nosfx)
                fig.savefig(fig_dirpath / (maskname_nosfx + '.png'))
                
                # stack mask on 'caax cell stack' and save image
                caax_cell_img = BioImage(caax_cell_path, reader=bioio_ome_tiff.Reader).data
                mask_refined = np.concatenate([caax_cell_img, mask_refined], axis=1)                
                OmeTiffWriter.save(mask_refined, refinedmaskpath)
        else:
            print(f'{maskname_nosfx} missing')

print('done!')
